# Forecasting Multi-Step

Previsoes para multiplos horizontes temporais.

## Conteudo
1. Recursive Forecasting
2. Direct Forecasting  
3. Comparacao

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb
from src.features.feature_engineering import FeatureEngineer
from src.models.evaluation import ModelEvaluator
print('Setup OK')

Setup OK


In [2]:
df = pd.read_parquet('../data/processed/processed_data.parquet')
fe = FeatureEngineer()
df_features = fe.create_all_features(df)
exclude_cols = ['timestamp', 'consumption_mw', 'region', 'holiday_name']
for col in df_features.columns:
    if pd.api.types.is_datetime64_any_dtype(df_features[col]) or df_features[col].dtype == 'object':
        if col not in exclude_cols: exclude_cols.append(col)
features = [c for c in df_features.columns if c not in exclude_cols]
df_sorted = df_features.sort_values('timestamp').reset_index(drop=True)
test_start = int(0.85 * len(df_sorted))
X_test = df_sorted.iloc[test_start:][features]
y_test = df_sorted.iloc[test_start:]['consumption_mw']
print(f'Test: {len(X_test):,} samples')

Creating features...
  - Temporal features
  - Lag features
  - Rolling features
  - Interaction features
  - Removed 240 rows with NaN
Test: 26,245 samples


In [ ]:
# Direct Forecasting
print('Direct Forecasting (treinar modelo para cada horizonte)...')
direct_results = {}
df_train = df_sorted.iloc[:int(0.70*len(df_sorted))]
evaluator = ModelEvaluator()
horizons = [1, 6, 12, 24]

for h in horizons:
    y_train_h = df_train['consumption_mw'].shift(-h).dropna()
    X_train_h = df_train[features].iloc[:len(y_train_h)]
    model_h = xgb.XGBRegressor(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=42, n_jobs=-1, verbosity=0)
    model_h.fit(X_train_h.values, y_train_h.values)
    y_test_h = y_test.shift(-h).dropna()
    X_test_h = X_test.iloc[:len(y_test_h)]
    predictions = model_h.predict(X_test_h.values)
    metrics = evaluator.calculate_metrics(y_test_h.values, predictions)
    direct_results[h] = metrics['mape']
    print(f'  Horizonte {h}h: MAPE = {metrics["mape"]:.2f}%')
    joblib.dump(model_h, f'../data/models/xgboost_horizon_{h}h.pkl')

print('\nModelos multi-step salvos!')